# Authors’ BERT sequence example, with optional held-out evaluation

This restores the authors’ original demonstration: `bert-base-uncased`, the literal sequence `AGCTAGCTAGCT`, a 10-class evidential head, and a one-task Normal–Inverse-Gamma (NIG) regression head. The original notebook provides **no training dataset, split, trained task checkpoint, or training loop**. Its randomly initialized task heads demonstrate interfaces and losses; their single-sequence outputs are not test-set results. `bert-base-uncased` is a general-language BERT checkpoint, not a DNA-pretrained model.

The original model dimensions, sequence, classification loss settings, regression loss combination, illustrative target, and training-variance value are retained. Stable softplus, inference mode, and consistent target scaling correct numerical/inference errors. The final section evaluates **actual held-out labeled sequences using your already-trained checkpoints** and makes no replacement dataset.

Install the optional dependency with `python -m pip install transformers`. Obtain the authors’ `bert-base-uncased` model/tokenizer files through your usual Hugging Face setup first. This notebook defaults to local cached files and never downloads a checkpoint automatically. Set `RUN_PRETRAINED_DEMO = True` when the cache is available. Held-out evaluation is independently controlled by the file paths near the end.


In [ ]:
from contextlib import contextmanager
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from mlguess.torch.class_losses import relu_evidence, edl_digamma_loss, edl_log_loss, edl_mse_loss
from mlguess.uncertainty import dirichlet_uncertainties, nig_uncertainties, inverse_scale_moments
from mlguess.evaluation import evaluate_classifier, evaluate_nig

PRETRAINED_NAME = "bert-base-uncased"  # authors' original model/tokenizer
LOCAL_FILES_ONLY = True
RUN_PRETRAINED_DEMO = False
DEVICE = torch.device("cpu")

@contextmanager
def inference_context(model):
    """Deterministic inference; restore each module's original training flag."""
    states = {module: module.training for module in model.modules()}
    try:
        model.eval()
        with torch.inference_mode():
            yield
    finally:
        for module, training in states.items():
            module.training = training


def load_tokenizer():
    from transformers import BertTokenizer
    return BertTokenizer.from_pretrained(PRETRAINED_NAME, local_files_only=LOCAL_FILES_ONLY)


## Original 10-class model

The CLS hidden state feeds the original linear head. Dirichlet moments obey `total = aleatoric + epistemic`; the Dempster–Shafer score `K / sum(alpha)` is reported separately and is not a variance. Uncertainty methods return detached NumPy arrays.


In [ ]:
class DNABert(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        from transformers import BertModel
        self.n_classes = n_classes
        self.bert = BertModel.from_pretrained(PRETRAINED_NAME, local_files_only=LOCAL_FILES_ONLY)
        self.fc = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        return self.fc(outputs.last_hidden_state[:, 0, :])

    def predict_uncertainty(self, input_ids, attention_mask, token_type_ids=None):
        with inference_context(self):
            logits = self(input_ids, attention_mask, token_type_ids)
            alpha = (relu_evidence(logits) + 1).cpu().numpy()
        probability, aleatoric, epistemic, _ = dirichlet_uncertainties(alpha)
        dst = self.n_classes / alpha.sum(axis=-1, keepdims=True)
        return probability, dst, aleatoric, epistemic


In [ ]:
num_classes = 10
# Original settings: no alternative classifier or generated training data.
dna_sequence = "AGCTAGCTAGCT"
loss_name = "digamma"
annealing_coefficient = 10.
epoch = 0

def classification_criterion(name):
    choices = {"digamma": edl_digamma_loss, "log": edl_log_loss, "mse": edl_mse_loss}
    if name not in choices:
        raise ValueError("Choose digamma, log, or mse")
    return choices[name]

if RUN_PRETRAINED_DEMO:
    tokenizer = load_tokenizer()
    classifier = DNABert(n_classes=num_classes).to(DEVICE).eval()
    inputs = {name: value.to(DEVICE) for name, value in tokenizer(dna_sequence, return_tensors="pt").items()}
    outputs = classifier(**inputs)
    prob, dst, aleatoric, epistemic = classifier.predict_uncertainty(**inputs)
    y_true_hot = torch.tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=torch.float32, device=DEVICE)
    classification_loss = classification_criterion(loss_name)(
        outputs, y_true_hot, epoch, num_classes, annealing_coefficient, device=DEVICE)
    display(pd.DataFrame({"class": np.arange(num_classes), "probability": prob[0],
                          "aleatoric_variance": aleatoric[0], "epistemic_variance": epistemic[0]}))
    print("Illustrative loss (untrained task head):", classification_loss.item(), "DST:", dst.item())
    # classification_loss.backward() is possible; this is not a training experiment.
else:
    print("Original single-sequence classification demonstration not run: RUN_PRETRAINED_DEMO=False.")


## Original one-task NIG model

The original head emits `(mu, nu, alpha, beta)`. `softplus(x)` evaluates the original `log(exp(x) + 1)` transformation stably. The original `training_var=[0.0033433]` is retained as metadata. A variance alone does not identify the mean offset or a complete target transformation: without a fitted scaler, predictions and variances stay in **model target units**. Passing the scaler fitted during training transforms means once and variances by its squared inverse slope. It must not be fitted on the test set.

For one NIG model, observation variance is `beta / (alpha - 1)`, mean uncertainty is `beta / (nu * (alpha - 1))`, and their sum is the predictive variance.


In [ ]:
class LinearNormalGamma(nn.Module):
    def __init__(self, in_chanels, out_channels):
        super().__init__()
        self.linear = nn.Linear(in_chanels, out_channels * 4)

    def evidence(self, x):
        return torch.nn.functional.softplus(x)

    def forward(self, x):
        pred = self.linear(x).view(x.shape[0], -1, 4)
        mu, logv, logalpha, logbeta = [value.squeeze(-1) for value in torch.split(pred, 1, dim=-1)]
        # Positive finite-variance parameters, including floating-point extremes.
        tiny = torch.finfo(pred.dtype).tiny
        eps = torch.finfo(pred.dtype).eps
        return (mu, self.evidence(logv).clamp_min(tiny),
                (self.evidence(logalpha) + 1).clamp_min(1 + eps),
                self.evidence(logbeta).clamp_min(tiny))


class DNABertRegressor(nn.Module):
    def __init__(self, n_tasks, training_var=(1.0,)):
        super().__init__()
        from transformers import BertModel
        self.output_size = n_tasks
        self.bert = BertModel.from_pretrained(PRETRAINED_NAME, local_files_only=LOCAL_FILES_ONLY)
        self.fc = LinearNormalGamma(self.bert.config.hidden_size, self.output_size)
        self.training_var = list(training_var)  # retain authors' metadata, not a complete scaler

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        return self.fc(outputs.last_hidden_state[:, 0, :])

    def predict_uncertainty(self, input_ids, attention_mask, token_type_ids=None, y_scaler=None):
        with inference_context(self):
            parameters = [value.cpu().numpy() for value in self(input_ids, attention_mask, token_type_ids)]
        mu, aleatoric, epistemic, _ = nig_uncertainties(*parameters)
        return inverse_scale_moments(mu, aleatoric, epistemic, y_scaler)


### Original regression loss demonstration

The authors’ original definitions and `tol=1e-8` are retained below, including the sum of marginal NLL, the evidence regularizer (factor `0.1`), and modified MSE. The only edit within these definitions constructs π with the prediction tensor’s dtype and device. The target `1.45456457` remains the original illustrative value; it is not a test dataset.


In [ ]:
tol = 1e-8

def modified_mse(gamma, nu, alpha, beta, target, reduction='mean'):
    """
    Lipschitz MSE loss of the "Improving evidential deep learning via multitask learning."

    Reference: https://www.mit.edu/~amini/pubs/pdf/deep-evidential-regression.pdf
    Source: https://github.com/deargen/MT-ENet/tree/468822188f52e517b1ee8e386eea607b2b7d8829

    Args:
        gamma ([FloatTensor]): the output of the ENet.
        nu ([FloatTensor]): the output of the ENet.
        alpha ([FloatTensor]): the output of the ENet.
        beta ([FloatTensor]): the output of the ENet.
        target ([FloatTensor]): true labels.
        reduction (str, optional): . Defaults to 'mean'.
    Returns:
        [FloatTensor]: The loss value. 
    """
    mse = (gamma-target)**2
    c = get_mse_coef(gamma, nu, alpha, beta, target).detach()
    mod_mse = mse*c
    
    if reduction == 'mean': 
        return mod_mse.mean()
    elif reduction == 'sum':
        return mod_mse.sum()
    else:
        return mod_mse

def get_mse_coef(gamma, nu, alpha, beta, y):
    """
    Return the coefficient of the MSE loss for each prediction.
    By assigning the coefficient to each MSE value, it clips the gradient of the MSE
    based on the threshold values U_nu, U_alpha, which are calculated by check_mse_efficiency_* functions.

    Reference: https://www.mit.edu/~amini/pubs/pdf/deep-evidential-regression.pdf
    Source: https://github.com/deargen/MT-ENet/tree/468822188f52e517b1ee8e386eea607b2b7d8829

    Args:
        gamma ([FloatTensor]): the output of the ENet.
        nu ([FloatTensor]): the output of the ENet.
        alpha ([FloatTensor]): the output of the ENet.
        beta ([FloatTensor]): the output of the ENet.
        y ([FloatTensor]): true labels.
    Returns:
        [FloatTensor]: [0.0-1.0], the coefficient of the MSE for each prediction.
    """
    alpha_eff = check_mse_efficiency_alpha(nu, alpha, beta)
    nu_eff = check_mse_efficiency_nu(gamma, nu, alpha, beta)
    delta = (gamma - y).abs()
    min_bound = torch.min(nu_eff, alpha_eff).min()
    c = (min_bound.sqrt()/(delta + tol)).detach()
    return torch.clip(c, min=False, max=1.)


def check_mse_efficiency_alpha(nu, alpha, beta):
    """
    Check the MSE loss (gamma - y)^2 can make negative gradients for alpha, which is
    a pseudo observation of the normal-inverse-gamma. We can use this to check the MSE
    loss can success(increase the pseudo observation, alpha).

    Reference: https://www.mit.edu/~amini/pubs/pdf/deep-evidential-regression.pdf
    Source: https://github.com/deargen/MT-ENet/tree/468822188f52e517b1ee8e386eea607b2b7d8829

    Args:
        nu (torch.Tensor): nu output value of the evidential network
        alpha (torch.Tensor): alpha output value of the evidential network
        beta (torch.Tensor): beta output value of the evidential network

    Return:
        partial f / partial alpha(numpy.array) 
        where f => the NLL loss (BayesianDTI.loss.MarginalLikelihood)
    
    """
    right = (torch.exp((torch.digamma(alpha+0.5)-torch.digamma(alpha))) - 1)*2*beta*(1+nu) / (nu + 1e-8)
    return right.detach()


def check_mse_efficiency_nu(gamma, nu, alpha, beta):
    """
    Check the MSE loss (gamma - y)^2 can make negative gradients for nu, which is
    a pseudo observation of the normal-inverse-gamma. We can use this to check the MSE
    loss can success(increase the pseudo observation, nu).

    Reference: https://www.mit.edu/~amini/pubs/pdf/deep-evidential-regression.pdf
    Source: https://github.com/deargen/MT-ENet/tree/468822188f52e517b1ee8e386eea607b2b7d8829

    Args:
        gamma (torch.Tensor): gamma output value of the evidential network
        nu (torch.Tensor): nu output value of the evidential network
        alpha (torch.Tensor): alpha output value of the evidential network
        beta (torch.Tensor): beta output value of the evidential network
    
    Return:
        partial f / partial nu(torch.Tensor) 
        where f => the NLL loss (BayesianDTI.loss.MarginalLikelihood)
    """
    gamma, nu, alpha, beta = gamma.detach(), nu.detach(), alpha.detach(), beta.detach()
    nu_1 = (nu + 1) / (nu + tol)
    return beta * nu_1 / (alpha + tol)

        
class EvidentialMarginalLikelihood(torch.nn.modules.loss._Loss):
    """
    Marginal likelihood error of prior network.
    The target value is not a distribution (mu, std), but a just value.
    
    This is a negative log marginal likelihood, with integral mu and sigma.

    Reference: https://www.mit.edu/~amini/pubs/pdf/deep-evidential-regression.pdf
    Source: https://github.com/deargen/MT-ENet/tree/468822188f52e517b1ee8e386eea607b2b7d8829
    """
    def __init__(self, size_average=None, reduce=None, reduction: str = 'mean'):
        super(EvidentialMarginalLikelihood, self).__init__(size_average, reduce, reduction)
    
    def forward(self, gamma: torch.Tensor, nu: torch.Tensor, alpha: torch.Tensor, beta: torch.Tensor,
                target: torch.Tensor) -> torch.Tensor:
        """
        Args:
            gamma (torch.Tensor): gamma output value of the evidential network
            nu (torch.Tensor): nu output value of the evidential network
            alpha (torch.Tensor): alpha output value of the evidential network
            beta (torch.Tensor): beta output value of the evidential network
            target (torch.Tensor): target value
            
        Return:
            (Tensor) Negative log marginal likelihood of EvidentialNet
                p(y|m) = Student-t(y; gamma, (beta(1+nu))/(nu*alpha) , 2*alpha)
                then, the negative log likelihood is (CAUTION QUITE COMPLEX!)
                NLL = -log(p(y|m)) =
                    log(3.14/nu)*0.5 - alpha*log(2*beta*(1 + nu)) + (alpha + 0.5)*log( nu(target - gamma)^2 + 2*beta(1 + nu) )
                    + log(GammaFunc(alpha)/GammaFunc(alpha + 0.5))
        """
        pi = nu.new_tensor(np.pi)
        x1 = torch.log(pi/(nu + tol))*0.5
        x2 = -alpha*torch.log(2.*beta*(1.+ nu) + tol)
        x3 = (alpha + 0.5)*torch.log( nu*(target - gamma)**2 + 2.*beta*(1. + nu) + tol)
        x4 = torch.lgamma(alpha + tol) - torch.lgamma(alpha + 0.5 + tol)
        if self.reduction == 'mean': 
            return (x1 + x2 + x3 + x4).mean()
        elif self.reduction == 'sum':
            return (x1 + x2 + x3 + x4).sum()
        else:
            return x1 + x2 + x3 + x4

class EvidenceRegularizer(torch.nn.modules.loss._Loss):
    """
    Regularization for the regression prior network.
    If self.factor increases, the model output the wider(high confidence interval) predictions.

    Reference: https://www.mit.edu/~amini/pubs/pdf/deep-evidential-regression.pdf
    Source: https://github.com/deargen/MT-ENet/tree/468822188f52e517b1ee8e386eea607b2b7d8829
    """
    def __init__(self, size_average=None, reduce=None, reduction: str = 'mean', factor=0.1):
        super(EvidenceRegularizer, self).__init__(size_average, reduce, reduction)
        self.factor = factor
    
    def forward(self, gamma: torch.Tensor, nu: torch.Tensor, alpha: torch.Tensor,
                target: torch.Tensor) -> torch.Tensor:
        """
        Args:
            gamma (torch.Tensor): gamma output value of the evidential network
            nu (torch.Tensor): nu output value of the evidential network
            alpha (torch.Tensor): alpha output value of the evidential network
            target (torch.Tensor): target value

        Return:
            (Tensor) prior network regularization
            Loss = |y - gamma|*(2*nu + alpha) * factor
            
        """
        loss_value =  torch.abs(target - gamma)*(2*nu + alpha) * self.factor
        if self.reduction == 'mean': 
            return loss_value.mean()
        elif self.reduction == 'sum':
            return loss_value.sum()
        else:
            return loss_value


In [ ]:
if RUN_PRETRAINED_DEMO:
    regressor = DNABertRegressor(n_tasks=1, training_var=[0.0033433]).to(DEVICE).eval()
    dna_sequence = "AGCTAGCTAGCT"
    inputs = {name: value.to(DEVICE) for name, value in tokenizer(dna_sequence, return_tensors="pt").items()}
    gamma, nu, alpha, beta = regressor(**inputs)
    mu, a, e = regressor.predict_uncertainty(**inputs)
    y_target = gamma.new_tensor([[1.45456457]])
    regression_loss = EvidentialMarginalLikelihood()(gamma, nu, alpha, beta, y_target)
    regression_loss += EvidenceRegularizer()(gamma, nu, alpha, y_target)
    regression_loss += modified_mse(gamma, nu, alpha, beta, y_target)
    mae = torch.nn.L1Loss()(y_target, gamma)
    display(pd.DataFrame({"mean": mu[:, 0], "aleatoric_variance": a[:, 0],
                          "epistemic_variance": e[:, 0]}))
    print("Illustrative loss:", regression_loss.item(), "illustrative MAE:", mae.item())
else:
    print("Original single-sequence regression demonstration not run: RUN_PRETRAINED_DEMO=False.")


## Additional evaluation on actual held-out data

Use a labeled CSV whose rows were held out before any preprocessing, training, hyperparameter choice, early stopping, or calibration. Sequence-related data should be split by the appropriate biological entity or source during training, so related sequences cannot leak between partitions. This section only reads the frozen test set; it does not train or select a model.

Provide a **trained** full `state_dict` (or a checkpoint dictionary with `model_state_dict`) matching the authors’ architecture above. For classification, the integer IDs `0..9` must match the class mapping used during training. For regression, supply the saved training-target scaler if targets were transformed. `REGRESSION_TARGETS_TRANSFORMED=False` explicitly means the checkpoint predicts physical target units. The retained numerical `training_var` alone cannot reconstruct a standardized target’s offset.

Leave unavailable paths as `None`; that evaluation will explicitly report that it was not run. No scores are generated from the untrained demonstration heads. Batch size is for inference only and does not change the authors’ model configuration.


In [ ]:
TEST_CSV = None                 # e.g. Path("data/held_out_sequences.csv")
CLASSIFIER_CHECKPOINT = None    # e.g. Path("checkpoints/classifier.pt")
REGRESSOR_CHECKPOINT = None     # e.g. Path("checkpoints/regressor.pt")
SEQUENCE_COLUMN = "sequence"
CLASS_COLUMN = "class_id"      # original 10-class head; IDs 0..9
REGRESSION_COLUMNS = ["target"]  # original one-task head
REGRESSION_TARGETS_TRANSFORMED = False
TARGET_SCALER_PATH = None       # joblib file saved during training, if transformed
INFERENCE_BATCH_SIZE = 16
RESULTS_DIR = Path("outputs/dna_original_evaluation")


In [ ]:
def predict_batches(model, tokenizer, sequences, batch_size=16):
    """Preserve sequence order and model state; return logits or NIG arrays."""
    sequences = list(sequences)
    if not sequences or any(not isinstance(value, str) or not value.strip() for value in sequences):
        raise ValueError("Provide nonempty sequence strings")
    if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError("batch_size must be a positive integer")
    device = next(model.parameters()).device
    batches = []
    with inference_context(model):
        for start in range(0, len(sequences), batch_size):
            encoded = tokenizer(sequences[start:start + batch_size], padding=True, truncation=True,
                                max_length=model.bert.config.max_position_embeddings, return_tensors="pt")
            values = model(**{name: value.to(device) for name, value in encoded.items()})
            if isinstance(values, tuple):
                batches.append(tuple(value.cpu().numpy() for value in values))
            else:
                batches.append(values.cpu().numpy())
    if isinstance(batches[0], tuple):
        return tuple(np.concatenate([batch[index] for batch in batches], axis=0) for index in range(4))
    return np.concatenate(batches, axis=0)


def restore_trained_state(model, path):
    state = torch.load(Path(path), map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE).eval()


In [ ]:
if TEST_CSV is None:
    test_data = None
    print("Held-out evaluation not run: set TEST_CSV and at least one trained checkpoint path.")
else:
    test_data = pd.read_csv(TEST_CSV)
    if SEQUENCE_COLUMN not in test_data or test_data.empty:
        raise ValueError("The test CSV must contain nonempty sequence rows")
    if test_data[SEQUENCE_COLUMN].isna().any():
        raise ValueError("Test sequences contain missing values")
    test_tokenizer = load_tokenizer()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
if test_data is not None and CLASSIFIER_CHECKPOINT is not None:
    test_classifier = restore_trained_state(DNABert(n_classes=10), CLASSIFIER_CHECKPOINT)
    test_logits = predict_batches(test_classifier, test_tokenizer, test_data[SEQUENCE_COLUMN], INFERENCE_BATCH_SIZE)
    test_alpha = np.maximum(test_logits, 0) + 1  # original ReLU evidence scheme
    test_probability, test_aleatoric, test_epistemic, _ = dirichlet_uncertainties(test_alpha)
    class_labels = pd.to_numeric(test_data[CLASS_COLUMN], errors="raise").to_numpy()
    if not np.isfinite(class_labels).all() or not np.equal(class_labels, np.floor(class_labels)).all():
        raise ValueError("Class labels must be integer IDs 0..9 from the training class mapping")
    class_labels = class_labels.astype(np.int64)
    classification_result = evaluate_classifier(class_labels, test_probability)
    display(pd.DataFrame([classification_result["metrics"]]))
    pd.DataFrame([classification_result["metrics"]]).to_csv(RESULTS_DIR / "classification_metrics.csv", index=False)
    predicted = test_probability.argmax(axis=1)
    predictions = pd.DataFrame({"row": np.arange(len(test_data)), "true_class": class_labels,
                                "predicted_class": predicted, "confidence": test_probability.max(axis=1),
                                "dst": 10 / test_alpha.sum(axis=1),
                                "aleatoric_trace": test_aleatoric.sum(axis=1),
                                "epistemic_trace": test_epistemic.sum(axis=1)})
    for index in range(10):
        predictions[f"probability_{index}"] = test_probability[:, index]
    predictions.to_csv(RESULTS_DIR / "classification_predictions.csv", index=False)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    im = axes[0].imshow(classification_result["confusion"], cmap="Blues")
    axes[0].set(xlabel="Predicted class", ylabel="True class", title="Held-out confusion matrix",
                xticks=np.arange(10), yticks=np.arange(10))
    fig.colorbar(im, ax=axes[0], label="Count")
    reliable = classification_result["reliability"].query("count > 0")
    axes[1].plot([0, 1], [0, 1], "--", color="gray")
    axes[1].plot(reliable.confidence, reliable.accuracy, "o-")
    axes[1].set(xlabel="Mean confidence", ylabel="Observed accuracy", title="Top-label reliability",
                xlim=(0, 1), ylim=(0, 1))
    correct = predicted == class_labels
    for mask, label in [(correct, "Correct"), (~correct, "Incorrect")]:
        if mask.any():
            axes[2].hist(test_epistemic[mask].sum(axis=1), bins=20, alpha=.65, label=label)
    axes[2].set(xlabel="Sum of class epistemic variances", ylabel="Count", title="Uncertainty and errors")
    axes[2].legend()
    fig.savefig(RESULTS_DIR / "classification_test.png", dpi=160)
    plt.show()
else:
    print("Classification test metrics and plots not produced: test CSV and trained checkpoint are required.")


In [ ]:
if test_data is not None and REGRESSOR_CHECKPOINT is not None:
    if len(REGRESSION_COLUMNS) != 1:
        raise ValueError("The authors' original regressor has one task; use exactly one target column")
    if REGRESSION_TARGETS_TRANSFORMED and TARGET_SCALER_PATH is None:
        raise ValueError("Load the target scaler fitted on training data; training_var alone is insufficient")
    if not REGRESSION_TARGETS_TRANSFORMED and TARGET_SCALER_PATH is not None:
        raise ValueError("A supplied target scaler requires REGRESSION_TARGETS_TRANSFORMED=True")
    target_scaler = None
    if TARGET_SCALER_PATH is not None:
        import joblib
        target_scaler = joblib.load(TARGET_SCALER_PATH)
    test_regressor = restore_trained_state(DNABertRegressor(n_tasks=1, training_var=[0.0033433]), REGRESSOR_CHECKPOINT)
    test_parameters = predict_batches(test_regressor, test_tokenizer, test_data[SEQUENCE_COLUMN], INFERENCE_BATCH_SIZE)
    regression_labels = test_data[REGRESSION_COLUMNS].to_numpy(dtype=float)
    regression_result = evaluate_nig(regression_labels, test_parameters, y_scaler=target_scaler)
    display(regression_result["metrics"])
    display(regression_result["coverage"])
    regression_result["metrics"].to_csv(RESULTS_DIR / "regression_metrics.csv")
    regression_result["coverage"].to_csv(RESULTS_DIR / "regression_coverage.csv", index=False)
    target = regression_labels[:, 0]
    mean = regression_result["mean"][:, 0]
    lower, upper = (bound[:, 0] for bound in regression_result["intervals"][.9])
    pd.DataFrame({"row": np.arange(len(test_data)), "target": target, "mean": mean,
                  "lower_90": lower, "upper_90": upper,
                  "aleatoric_variance": regression_result["aleatoric"][:, 0],
                  "epistemic_variance": regression_result["epistemic"][:, 0],
                  "pit": regression_result["pit"][:, 0]}).to_csv(RESULTS_DIR / "regression_predictions.csv", index=False)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
    axes[0, 0].scatter(target, mean, alpha=.6, s=14)
    limits = [min(target.min(), mean.min()), max(target.max(), mean.max())]
    axes[0, 0].plot(limits, limits, "--", color="gray")
    axes[0, 0].set(xlabel="Observed target", ylabel="Predicted mean", title="Held-out predictions")
    # Prediction ordering is only for display, not model selection or refitting.
    order = np.argsort(mean)
    rank = np.arange(len(order))
    axes[0, 1].fill_between(rank, lower[order], upper[order], alpha=.25, label="90% predictive interval")
    axes[0, 1].plot(rank, mean[order], label="Mean")
    axes[0, 1].scatter(rank, target[order], s=8, alpha=.5, label="Observed")
    axes[0, 1].set(xlabel="Test row ordered by predicted mean", ylabel="Target", title="Student-t predictive intervals")
    axes[0, 1].legend(fontsize=8)
    coverage = regression_result["coverage"]
    axes[0, 2].plot([0, 1], [0, 1], "--", color="gray")
    axes[0, 2].plot(coverage.nominal, coverage.observed, "o-")
    axes[0, 2].set(xlabel="Nominal coverage", ylabel="Observed coverage", title="Interval calibration",
                  xlim=(0, 1), ylim=(0, 1))
    axes[1, 0].hist(regression_result["pit"][:, 0], bins=np.linspace(0, 1, 11), density=True)
    axes[1, 0].axhline(1, linestyle="--", color="gray")
    axes[1, 0].set(xlabel="Predictive CDF at observed target", ylabel="Density", title="PIT histogram")
    axes[1, 1].scatter(np.sqrt(regression_result["aleatoric"][:, 0]),
                       np.sqrt(regression_result["epistemic"][:, 0]), s=14, alpha=.6)
    axes[1, 1].set(xlabel="Aleatoric SD", ylabel="Epistemic SD", title="Uncertainty components")
    axes[1, 2].scatter(np.sqrt(regression_result["total"][:, 0]), np.abs(target - mean), s=14, alpha=.6)
    axes[1, 2].set(xlabel="Predictive SD", ylabel="Absolute error", title="Uncertainty and errors")
    fig.savefig(RESULTS_DIR / "regression_test.png", dpi=160)
    plt.show()
else:
    print("Regression test metrics and plots not produced: test CSV and trained checkpoint are required.")


These intervals describe the predictive Student-t distribution, not a confidence interval for the mean. The displayed aleatoric and epistemic components are the chosen evidential model's decomposition; finite test-set coverage and calibration must be assessed from the actual results. Keep test predictions, class mapping, training-target scaler, checkpoint, split provenance, software versions, and random seeds with the experiment. Use validation data for any later calibration or model changes, then evaluate on a fresh held-out set.
